In [1]:
import numpy as np

In [2]:

class Perceptron:
    def __init__(self, learning_rate=0.1, n_iterations=100):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        """
        Train the perceptron model.
        X: Training data shape (n_samples, n_features)
        y: Target values shape (n_samples,)
        """
        n_samples, n_features = X.shape

        # 1. Initialize weights and bias to zeros
        self.weights = np.zeros(n_features)
        self.bias = 0

        # 2. Iterate through the dataset n_iterations times
        for _ in range(self.n_iterations):
            for idx, x_i in enumerate(X):

                # Calculate linear output: z = w.x + b
                linear_output = np.dot(x_i, self.weights) + self.bias

                # Apply Step Function
                y_predicted = self._step_function(linear_output)

                # Calculate update rule: (y_true - y_pred)
                update = self.lr * (y[idx] - y_predicted)

                # Update weights and bias
                # If update is 0 (correct prediction), weights don't change
                self.weights += update * x_i
                self.bias += update

    def predict(self, X):
        """Predict class labels for samples in X"""
        linear_output = np.dot(X, self.weights) + self.bias
        return self._step_function(linear_output)

    def _step_function(self, x):
        """Binary step function: returns 1 if x >= 0, else 0"""
        return np.where(x >= 0, 1, 0)

# --- Application: Learning the AND Gate ---

if __name__ == "__main__":
    # Training Data (AND Logic Gate)
    # Input: [0,0], [0,1], [1,0], [1,1]
    X_train = np.array([
        [0, 0],
        [0, 1],
        [1, 0],
        [1, 1]
    ])

    # Target Output for AND (Only 1 if both inputs are 1)
    y_train = np.array([0, 0, 0, 1])

    # Initialize and Train
    p = Perceptron(learning_rate=0.1, n_iterations=10)
    p.fit(X_train, y_train)

    # Test
    predictions = p.predict(X_train)

    print("AND Gate Training Results:")
    print(f"Weights: {p.weights}")
    print(f"Bias: {p.bias}")
    print(f"Predictions: {predictions}")
    print(f"Ground Truth: {y_train}")

AND Gate Training Results:
Weights: [0.2 0.1]
Bias: -0.20000000000000004
Predictions: [0 0 0 1]
Ground Truth: [0 0 0 1]


For the input [1, 1], the target is 1.
Eventually, the weights and bias settled on values (e.g., $w=[0.1, 0.2], b=-0.2$) such that only the input [1, 1] results in a positive sum ($z \geq 0$), while [0, 1], [1, 0], and [0, 0] result in negative sums ($z < 0$), correctly replicating the AND gate logic.



### Multi Layer Perceptron

In [3]:
# --- Helper Functions ---

def sigmoid(x):
    """
    Activation function: Maps any value to a value between 0 and 1.
    Formula: 1 / (1 + e^-x)
    """
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    """
    The derivative of the sigmoid function.
    Used during backpropagation to calculate gradients.
    If y = sigmoid(x), then y' = y * (1 - y).
    """
    return x * (1 - x)

# --- Multilayer Perceptron Class ---

class MultilayerPerceptron:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.1):
        self.learning_rate = learning_rate

        # 1. Initialize Weights with random values
        # We cannot initialize to 0 (like the single perceptron) because
        # that would prevent "symmetry breaking" (all neurons would learn the same thing).
        self.weights_input_hidden = np.random.uniform(size=(input_size, hidden_size))
        self.weights_hidden_output = np.random.uniform(size=(hidden_size, output_size))

        # Initialize Biases
        self.bias_hidden = np.random.uniform(size=(1, hidden_size))
        self.bias_output = np.random.uniform(size=(1, output_size))

    def forward(self, X):
        """
        Forward Pass: Propagate inputs through the network to get a prediction.
        """
        # --- Step 1: Input -> Hidden Layer ---
        # Calculate weighted sum: z = X . W + b
        self.hidden_input = np.dot(X, self.weights_input_hidden) + self.bias_hidden
        # Apply activation function
        self.hidden_output = sigmoid(self.hidden_input)

        # --- Step 2: Hidden -> Output Layer ---
        # Calculate weighted sum from hidden nodes
        self.final_input = np.dot(self.hidden_output, self.weights_hidden_output) + self.bias_output
        # Apply activation function for final prediction
        self.final_output = sigmoid(self.final_input)

        return self.final_output

    def backward(self, X, y, output):
        """
        Backward Pass: Calculate errors and update weights (Backpropagation).
        """
        # --- Step 1: Calculate Output Layer Error ---
        # Error = Target - Prediction
        error_output = y - output

        # Calculate Delta (Gradient) for Output Layer
        # Delta = Error * Derivative of Activation Function
        # This tells us: "How much did the output activation contribute to the error?"
        delta_output = error_output * sigmoid_derivative(output)

        # --- Step 2: Calculate Hidden Layer Error ---
        # We propagate the output delta BACKWARDS to the hidden layer.
        # How much did the hidden layer weights contribute to the output error?
        error_hidden = delta_output.dot(self.weights_hidden_output.T)

        # Calculate Delta (Gradient) for Hidden Layer
        delta_hidden = error_hidden * sigmoid_derivative(self.hidden_output)

        # --- Step 3: Update Weights and Biases (Gradient Descent) ---
        # Weight_new = Weight_old + (Input * Delta * Learning_Rate)

        # Update Hidden-to-Output weights
        self.weights_hidden_output += self.hidden_output.T.dot(delta_output) * self.learning_rate
        self.bias_output += np.sum(delta_output, axis=0, keepdims=True) * self.learning_rate

        # Update Input-to-Hidden weights
        self.weights_input_hidden += X.T.dot(delta_hidden) * self.learning_rate
        self.bias_hidden += np.sum(delta_hidden, axis=0, keepdims=True) * self.learning_rate

    def train(self, X, y, epochs=10000):
        """
        Training loop: Runs Forward and Backward pass 'epochs' times.
        """
        for i in range(epochs):
            # 1. Forward Pass
            output = self.forward(X)

            # 2. Backward Pass (Train)
            self.backward(X, y, output)

            # Optional: Print loss every 1000 epochs to monitor progress
            if (i) % 1000 == 0:
                loss = np.mean(np.square(y - output)) # Mean Squared Error
                print(f"Epoch {i}, Loss: {loss:.4f}")

# --- Application: Solving XOR ---

if __name__ == "__main__":
    # The XOR Problem Data
    # Inputs: [0,0], [0,1], [1,0], [1,1]
    X = np.array([
        [0, 0],
        [0, 1],
        [1, 0],
        [1, 1]
    ])

    # Target Output: [0], [1], [1], [0]
    # Note: reshape(-1, 1) ensures the target is a column vector
    y = np.array([[0], [1], [1], [0]])

    print("Training Multilayer Perceptron on XOR data...")

    # Initialize MLP
    # Input Nodes: 2 (for the two binary inputs)
    # Hidden Nodes: 2 (minimum required to solve XOR)
    # Output Nodes: 1 (binary result)
    mlp = MultilayerPerceptron(input_size=2, hidden_size=2, output_size=1, learning_rate=0.1)

    # Train the model
    mlp.train(X, y, epochs=10000)

    # Test the trained model
    print("\n--- Final Predictions ---")
    predictions = mlp.forward(X)

    for input_val, pred, true_val in zip(X, predictions, y):
        # We round the prediction to get a clean 0 or 1, but print the raw probability too
        print(f"Input: {input_val} | Prediction: {pred[0]:.4f} ({int(np.round(pred[0]))}) | True: {true_val[0]}")

Training Multilayer Perceptron on XOR data...
Epoch 0, Loss: 0.3221
Epoch 1000, Loss: 0.2500
Epoch 2000, Loss: 0.2499
Epoch 3000, Loss: 0.2498
Epoch 4000, Loss: 0.2494
Epoch 5000, Loss: 0.2463
Epoch 6000, Loss: 0.2244
Epoch 7000, Loss: 0.1799
Epoch 8000, Loss: 0.1196
Epoch 9000, Loss: 0.0297

--- Final Predictions ---
Input: [0 0] | Prediction: 0.1052 (0) | True: 0
Input: [0 1] | Prediction: 0.8928 (1) | True: 1
Input: [1 0] | Prediction: 0.8934 (1) | True: 1
Input: [1 1] | Prediction: 0.1223 (0) | True: 0


#### Key Differences from the Single Perceptron Code.
1. Classes vs. Raw Loops: This code is structured within a class to manage the multiple weight matrices (weights_input_hidden and weights_hidden_output).
2. Derivatives: Notice the sigmoid_derivative function. The single perceptron uses a simple subtraction (y - y_hat). The MLP must use calculus (derivatives) because the hidden layer error is not directly observable; we have to infer it mathematically.
3. Initialization: In the single perceptron, we initialized weights to zeros. Here, *_np.random.uniform_* is crucial. If you initialize all MLP weights to zero, all neurons in the hidden layer will calculate the exact same gradient and learn the exact same features, effectively reducing your neural network to a single neuron.